In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
# --- Our version ---
import os
import sys
from langchain_google_genai import ChatGoogleGenerativeAI
from openai import OpenAI  # Groq and OpenRouter use OpenAI-compatible endpoints

In [3]:
# --- Our version ---
# No single MODEL constant — we support multiple providers, chosen per call.
# No DB_NAME either, since we don't persist to disk (DECISIONS.md D-005).

load_dotenv(override=True)

gemini_api_key = os.getenv('GEMINI_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

In [ ]:
import os
os.chdir("..")  # move from notebooks/ up to the pitchwise/ root
print(os.getcwd())  # confirm it's now .../pitchwise

In [5]:
# --- Our version ---
# We don't persist to disk (DECISIONS.md D-005), so instead of reconnecting
# to a saved database, we run our own ingestion pipeline fresh via ingest.py

sys.path.append("..")
from ingest import ingest

vectorstore = ingest()

Loaded 17 documents from knowledge-base
Split into 81 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store built: 81 chunks embedded, 384 dimensions


In [6]:
# --- Our version ---
retriever = vectorstore.as_retriever()

# ChatGoogleGenerativeAI is LangChain's wrapper for Gemini — same .invoke()
# interface as ChatOpenAI, so everything downstream stays consistent
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0,
    google_api_key=gemini_api_key
)

In [7]:
retriever.invoke("Who is Don Bradman?")

[Document(id='17974d89-c365-4dbb-9e76-b16482e13166', metadata={'source': 'knowledge-base/players/don-bradman.md', 'header_1': 'Don Bradman', 'header_2': 'Career Overview', 'doc_type': 'players'}, page_content='Don Bradman is universally regarded as the greatest batter in the history of cricket. He played 52 Test matches for Australia across two decades, spanning the era before and after World War II, and captained Australia for much of his later career.'),
 Document(id='d343b6a7-6b57-41d0-acba-95b8d14ab5ea', metadata={'doc_type': 'players', 'source': 'knowledge-base/players/don-bradman.md', 'header_1': 'Don Bradman'}, page_content='**Full name:** Sir Donald George Bradman\n**Born:** 27 August 1908, Cootamundra, New South Wales, Australia\n**Died:** 25 February 2001, Adelaide, Australia\n**Role:** Batter\n**Batting style:** Right-handed\n**National side:** Australia (1928–1948, retired)'),
 Document(id='20ca08de-5e23-42ef-a5ed-32dddbc15b11', metadata={'source': 'knowledge-base/players/d

In [ ]:
llm.invoke("Who is Don Bradman?")

In [9]:
# --- Our version ---
SYSTEM_PROMPT_TEMPLATE = """
You are Pitchwise, a knowledgeable, friendly cricket assistant.
You are chatting with a user about cricket — players, formats, tournaments, and the laws of the game.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [10]:
# --- Our version ---
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("What is Don Bradman's exact career Test batting average?", [])

In [12]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import time

start = time.time()
response = answer_question("What is Don Bradman's exact career Test batting average?", [])
elapsed = time.time() - start
print(f"Took {elapsed:.1f} seconds")

In [ ]:
import time

# Time retrieval alone
start = time.time()
docs = retriever.invoke("What is Don Bradman's exact career Test batting average?")
print(f"Retrieval: {time.time() - start:.1f}s")

# Time the LLM call alone (using context we just retrieved)
context = "\n\n".join(doc.page_content for doc in docs)
system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)

start = time.time()
response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content="What is Don Bradman's exact career Test batting average?")])
print(f"LLM call: {time.time() - start:.1f}s")

In [15]:
# --- Testing Groq for the same query ---
from openai import OpenAI

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

start = time.time()
groq_response = groq.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "What is Don Bradman's exact career Test batting average?"}
    ]
)
print(f"Groq call: {time.time() - start:.1f}s")
print(groq_response.choices[0].message.content)

Groq call: 1.0s
Don Bradman’s career Test batting average is **99.94**. This figure comes from his 6,996 runs in 52 Tests and remains the highest ever recorded for a player with a substantial number of innings.


In [ ]:
# --- Full provider/model speed comparison ---
import time

question = "What is Don Bradman's exact career Test batting average?"
# system_prompt already built from earlier retrieval — reusing it here

candidates = {
    "gemini-3.6-flash": ("gemini", "gemini-3.6-flash"),
    "gemini-3.5-flash-lite": ("gemini", "gemini-3.5-flash-lite"),
    "groq: gpt-oss-120b": ("groq", "openai/gpt-oss-120b"),
    "groq: gpt-oss-20b": ("groq", "openai/gpt-oss-20b"),
    "groq: qwen3.8-27b": ("groq", "qwen/qwen3.8-27b"),
    "openrouter: minimax-m3": ("openrouter", "minimax/minimax-m3:free"),
    "openrouter: glm-5.2": ("openrouter", "z-ai/glm-5.2:free"),
}

groq_client = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
openrouter_client = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")

speed_results = {}

for label, (provider, model) in candidates.items():
    try:
        start = time.time()

        if provider == "gemini":
            temp_llm = ChatGoogleGenerativeAI(model=model, temperature=0, google_api_key=gemini_api_key)
            response = temp_llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
            answer_text = response.content

        elif provider == "groq":
            response = groq_client.chat.completions.create(
                model=model,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": question}]
            )
            answer_text = response.choices[0].message.content

        elif provider == "openrouter":
            response = openrouter_client.chat.completions.create(
                model=model,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": question}]
            )
            answer_text = response.choices[0].message.content

        elapsed = time.time() - start
        speed_results[label] = {"time": elapsed, "answer": answer_text}
        print(f"{label}: {elapsed:.1f}s")

    except Exception as e:
        print(f"{label}: FAILED — {e}")
        speed_results[label] = {"time": None, "answer": None}

print("\n--- Ranked by speed ---")
ranked = sorted(
    [(k, v["time"]) for k, v in speed_results.items() if v["time"] is not None],
    key=lambda x: x[1]
)
for label, t in ranked:
    print(f"{t:.1f}s — {label}")

In [ ]:
# --- Testing the fallback mechanism (DECISIONS.md D-008) ---
import answer
from ingest import ingest

# Build a vector store to test against (if you don't already have one
# named `store` in this notebook session)
store = ingest()

real_groq_key = answer.GROQ_API_KEY
answer.GROQ_API_KEY = "intentionally-invalid-key-for-testing"

test_question = "What is Don Bradman's exact career Test batting average?"

result = answer.answer_question(
    store,
    test_question,
    model_label="Groq: qwen3.8-27b (fastest, default)"
)

print(f"Q: {test_question}")
print(f"A: {result}")

answer.GROQ_API_KEY = real_groq_key
print("\nGroq key restored.")